# Bulk Verification

- Apply standard error and performance metrics across all model predictions
- Compared with same metrics from GSO optical flow predictions

In [1]:
import xgboost as xgb
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from pathlib import Path
import numpy as np

from sklearn.metrics import mean_squared_error
from skimage.metrics import structural_similarity as ssim

import sys
sys.path.append('/home/548/cd3022/repos/solar-nowcast/modules')
from gso_preprocess import datetime_to_lead_time

In [2]:
model = xgb.XGBRegressor()

In [4]:
model.load_model("/scratch/er8/cd3022/xgb_models/alldata_model.json")

test_list = []
data_path = Path('/scratch/er8/cd3022/xgb_datasets/')
for f in data_path.glob('*all_training*'):
    df = pd.read_parquet(f)

    split_index = int(len(df) * 0.8)
    test = df.iloc[split_index:]

    test_list.append(test)


all_test = pd.concat(test_list)

# REMOVE FUTURE VALUES FROM THE TRAINING DATA

X = all_test.drop(columns=['cloud_optical_depth'])
for n in range(1, 7):
    X = X.drop(columns=[f'cloud_optical_depth_t{n}'])

# SELECT FORECAST TIME HORIZON
# (for alldata, times are aligned so have been reduced to BARRA 1hr time steps)
y = all_test['cloud_optical_depth_t1']

y = data_transform.log_transform(y)

df = X.join(y)
df['cloud_optical_depth'] = all_test['cloud_optical_depth']
df['forecast'] = model.predict(X)

ds_xgb = df.to_xarray()

/jobfs/165716340.gadi-pbs/ipykernel_942296/2461312106.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y[y == 0] = 0.1


In [ ]:
gso_files = []
for month in ['01', '03', '06', '09']:
    gso_path = Path(f'/scratch/er8/mm4602/gso/output/nwc_20240701_20251231/2025/{month}/05')
    gso_files.extend([f for f in gso_path.glob('*.nc')])

ds_gso = xr.open_mfdataset(
    gso_files,
    preprocess=datetime_to_lead_time,
    combine='nested',
    concat_dim='time',
    parallel=True,
    chunks='auto'
)

In [ ]:
ds_gso, ds_xgb = xr.align(
    ds_gso,
    ds_xgb,
    join="inner"
)

In [ ]:
obs = ds_xgb.cloud_optical_depth
fcst_xgb = ds_xgb.forecast
fcst_gso = ds_gso.sel(forecast_time='01:00:00').cloud_optical_depth

In [ ]:
mse_xgb = np.mean((fcst_xgb - obs) ** 2)
rmse_xgb = np.sqrt(mse_xgb)
print(f"XGBoost RMSE: {rmse_xgb.item()}")

mse_gso = np.mean((fcst_gso - obs) ** 2)
rmse_gso = np.sqrt(mse_gso)
rmse_gso = rmse_gso.compute()
print(f"Optical Flow RMSE: {rmse_gso.item()}")

In [ ]:
def compute_ssim(a, b):
    # Handle NaNs
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() == 0:
        return np.nan

    a_valid = a[mask]
    b_valid = b[mask]

    data_range = a_valid.max() - a_valid.min()
    if data_range == 0:
        return np.nan

    # Reshape back to 2D (required by SSIM)
    a2 = np.where(mask, a, 0)
    b2 = np.where(mask, b, 0)

    return ssim(a2, b2, data_range=data_range)

In [ ]:
ssim_xgb = xr.apply_ufunc(
    compute_ssim,
    obs,
    fcst_xgb,
    input_core_dims=[["latitude", "longitude"], ["latitude", "longitude"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

ssim_gso = xr.apply_ufunc(
    compute_ssim,
    obs,
    fcst_gso,
    input_core_dims=[["latitude", "longitude"], ["latitude", "longitude"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

In [ ]:
print(f"XGBoost SSIM: {ssim_xgb.mean().item()}")
print(f"Optical Flow SSIM: {ssim_gso.mean().compute().item()}")